# Transcript length and verification

Describe transcript length and verification categories across the three LLM arms.

## Configure analysis

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import chi2_contingency, f_oneway, kruskal

DATA_PATH = Path("..").resolve() / "data" / "Psy_Case_Vignette_Data_Prolific_analysis.xlsx"

VIGNETTES = ["A", "B", "C", "D", "E"]
LLM_ARMS = {1, 2, 3}
ARM_LABELS = {1: "ChatGPT", 2: "Copilot", 3: "Perplexity"}
ARM_ORDER = ["ChatGPT", "Copilot", "Perplexity"]
ARM_COLORS = {
    "ChatGPT": "#009E73",
    "Copilot": "#0072B2",
    "Perplexity": "#E69F00",
}

LENGTH_COLS = [f"vig_{v}_post_transcript_length" for v in VIGNETTES]
VALIDATION_COLS = [f"vig_{v}_post_transcript_validation" for v in VIGNETTES]

## Load and reshape transcript data

In [ ]:
df = pd.read_excel(DATA_PATH)

missing_cols = [
    col
    for col in LENGTH_COLS + VALIDATION_COLS + ["study_arm", "prolific_id"]
    if col not in df.columns
]
if missing_cols:
    raise KeyError(f"Missing columns: {missing_cols}")

llm = df.loc[df["study_arm"].isin(LLM_ARMS)].copy()
llm["Arm"] = pd.Categorical(
    llm["study_arm"].map(ARM_LABELS), categories=ARM_ORDER, ordered=True
)

long_parts = []
for vignette in VIGNETTES:
    length_col = f"vig_{vignette}_post_transcript_length"
    validation_col = f"vig_{vignette}_post_transcript_validation"
    long_parts.append(
        llm[["prolific_id", "Arm", length_col, validation_col]].rename(
            columns={length_col: "length", validation_col: "validation"}
        ).assign(vignette=vignette)
    )

long_df = pd.concat(long_parts, ignore_index=True)
long_df["validation"] = pd.to_numeric(long_df["validation"], errors="coerce")
long_df["length"] = pd.to_numeric(long_df["length"], errors="coerce")

participant_means = (
    long_df.groupby(["prolific_id", "Arm"], observed=True)["length"]
    .mean()
    .reset_index(name="mean_length")
)

print(f"File: {DATA_PATH.name}")
print(f"LLM participants: {llm['prolific_id'].nunique():,}")
print(f"Participant-vignette rows: {len(long_df):,}")
print("\nArm counts (participants):")
print(llm["Arm"].value_counts().sort_index().to_string())

## Summarize transcript length

In [ ]:
print("Overall (pooled participant × vignette rows)")
print(long_df["length"].describe().round(1).to_string())
print(f"\nMean words: {long_df['length'].mean():.1f}")
print(f"Median words: {long_df['length'].median():.1f}")

print("\nMean length by arm (pooled rows)")
by_arm = (
    long_df.groupby("Arm", observed=True)["length"]
    .agg(n="count", mean="mean", median="median", std="std")
    .round(1)
)
print(by_arm.to_string())

print("\nMean length by vignette and arm")
by_vignette_arm = (
    long_df.groupby(["vignette", "Arm"], observed=True)["length"]
    .agg(n="count", mean="mean", median="median")
    .round(1)
)
print(by_vignette_arm.to_string())

print("\nParticipant-level mean length by arm (one mean per participant)")
by_participant_arm = (
    participant_means.groupby("Arm", observed=True)["mean_length"]
    .agg(n="count", mean="mean", median="median", std="std")
    .round(1)
)
print(by_participant_arm.to_string())

### Compare transcript length across arms

In [ ]:
def arm_groups(frame: pd.DataFrame, value_col: str) -> list:
    return [
        frame.loc[frame["Arm"] == arm, value_col].dropna().to_numpy()
        for arm in ARM_ORDER
    ]


groups_participant = arm_groups(participant_means, "mean_length")
groups_pooled = arm_groups(long_df, "length")

kw_participant = kruskal(*groups_participant)
kw_pooled = kruskal(*groups_pooled)
anova_pooled = f_oneway(*groups_pooled)

print("Kruskal–Wallis — participant-level mean length across arms")
print(f"  H = {kw_participant.statistic:.3f}, p = {kw_participant.pvalue:.4g}")

print("\nKruskal–Wallis — pooled participant × vignette rows")
print(f"  H = {kw_pooled.statistic:.3f}, p = {kw_pooled.pvalue:.4g}")

print("\nOne-way ANOVA — pooled participant × vignette rows")
print(f"  F = {anova_pooled.statistic:.3f}, p = {anova_pooled.pvalue:.4g}")

### Plot transcript length

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

means_pooled = long_df.groupby("Arm", observed=True)["length"].mean().reindex(ARM_ORDER)
axes[0].bar(means_pooled.index, means_pooled.values, color=[ARM_COLORS[a] for a in ARM_ORDER])
axes[0].set_title("Mean transcript length by arm\n(pooled vignettes)")
axes[0].set_ylabel("Words")
axes[0].tick_params(axis="x", rotation=20)

means_participant = participant_means.groupby("Arm", observed=True)["mean_length"].mean().reindex(ARM_ORDER)
axes[1].bar(means_participant.index, means_participant.values, color=[ARM_COLORS[a] for a in ARM_ORDER])
axes[1].set_title("Mean transcript length by arm\n(participant-level means)")
axes[1].set_ylabel("Words")
axes[1].tick_params(axis="x", rotation=20)

fig.tight_layout()
plt.show()

## Summarize transcript verification

In [ ]:
validation_values = sorted(long_df["validation"].dropna().unique())
validation_values = [int(v) for v in validation_values]

print("Overall validation distribution (pooled participant × vignette rows)")
overall_counts = long_df["validation"].value_counts(dropna=False).sort_index()
overall_pct = (overall_counts / overall_counts.sum() * 100).round(1)
overall_table = pd.DataFrame({"n": overall_counts, "pct": overall_pct})
print(overall_table.to_string())

print("\nValidation distribution by arm (counts)")
by_arm_counts = pd.crosstab(long_df["Arm"], long_df["validation"], dropna=False)
by_arm_counts = by_arm_counts.reindex(index=ARM_ORDER, columns=validation_values, fill_value=0)
print(by_arm_counts.to_string())

print("\nValidation distribution by arm (% within arm)")
by_arm_pct = by_arm_counts.div(by_arm_counts.sum(axis=1), axis=0) * 100
print(by_arm_pct.round(1).to_string())

chi2_table = pd.crosstab(long_df["Arm"], long_df["validation"], dropna=True)
chi2_table = chi2_table.reindex(index=ARM_ORDER, fill_value=0)
chi2, p_chi2, dof, _ = chi2_contingency(chi2_table.to_numpy())
print(f"\nChi-square (validation × arm): chi2 = {chi2:.3f}, df = {dof}, p = {p_chi2:.4g}")

### Plot verification distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.ravel()

overall_plot = overall_counts.reindex(validation_values, fill_value=0)
axes[0].bar(
    [str(v) for v in validation_values],
    overall_plot.values,
    color="#555555",
)
axes[0].set_title("Overall validation distribution")
axes[0].set_xlabel("Validation code")
axes[0].set_ylabel("Count")

for ax, arm in zip(axes[1:], ARM_ORDER):
    arm_counts = by_arm_counts.loc[arm].reindex(validation_values, fill_value=0)
    ax.bar(
        [str(v) for v in validation_values],
        arm_counts.values,
        color=ARM_COLORS[arm],
    )
    ax.set_title(f"{arm}")
    ax.set_xlabel("Validation code")
    ax.set_ylabel("Count")

fig.suptitle("Transcript validation by LLM arm", y=1.02, fontsize=13)
fig.tight_layout()
plt.show()